### wap to implement the naive bayesian classifier for a sample training data set stroed as a CSV file . ccomputer the accuarcy of the classifier , considering few set test data set.

In [20]:
import pandas as pd
import numpy as np

In [21]:
# Load the dataset
df = pd.read_csv('../exp5/tennisdata.csv')
print("Dataset:")
print(df.head())
print("\nDataset shape:", df.shape)

Dataset:
    Outlook Temperature Humidity  Windy PlayTennis
0     Sunny         Hot     High  False         No
1     Sunny         Hot     High   True         No
2  Overcast         Hot     High  False        Yes
3     Rainy        Mild     High  False        Yes
4     Rainy        Cool   Normal  False        Yes

Dataset shape: (14, 5)


In [22]:
# Split the data into training and testing sets manually
np.random.seed(42)
indices = np.random.permutation(len(df))
test_size = int(0.3 * len(df))
test_indices = indices[:test_size]
train_indices = indices[test_size:]

X_train = df.drop('PlayTennis', axis=1).iloc[train_indices]
y_train = df['PlayTennis'].iloc[train_indices]
X_test = df.drop('PlayTennis', axis=1).iloc[test_indices]
y_test = df['PlayTennis'].iloc[test_indices]

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (10, 4)
Testing set shape: (4, 4)


In [23]:
# Train the Naive Bayes classifier
def train_naive_bayes(X_train, y_train):
    classes = y_train.unique()
    priors = {}
    likelihoods = {}
    
    for cls in classes:
        priors[cls] = len(y_train[y_train == cls]) / len(y_train)
        likelihoods[cls] = {}
        for feature in X_train.columns:
            likelihoods[cls][feature] = {}
            for value in X_train[feature].unique():
                count = len(X_train[(X_train[feature] == value) & (y_train == cls)])
                total = len(y_train[y_train == cls])
                likelihoods[cls][feature][value] = count / total if total > 0 else 0
    
    return priors, likelihoods, classes

priors, likelihoods, classes = train_naive_bayes(X_train, y_train)
print("Priors:", priors)
print("Likelihoods:", likelihoods)

Priors: {'No': 0.4, 'Yes': 0.6}
Likelihoods: {'No': {'Outlook': {'Rainy': 0.5, 'Sunny': 0.5, 'Overcast': 0.0}, 'Temperature': {'Cool': 0.25, 'Hot': 0.25, 'Mild': 0.5}, 'Humidity': {'Normal': 0.25, 'High': 0.75}, 'Windy': {np.True_: 0.75, np.False_: 0.25}}, 'Yes': {'Outlook': {'Rainy': 0.3333333333333333, 'Sunny': 0.3333333333333333, 'Overcast': 0.3333333333333333}, 'Temperature': {'Cool': 0.5, 'Hot': 0.16666666666666666, 'Mild': 0.3333333333333333}, 'Humidity': {'Normal': 0.6666666666666666, 'High': 0.3333333333333333}, 'Windy': {np.True_: 0.3333333333333333, np.False_: 0.6666666666666666}}}


In [24]:
# Prediction function
def predict_naive_bayes(sample, priors, likelihoods, classes):
    posteriors = {}
    for cls in classes:
        posterior = priors[cls]
        for feature, value in sample.items():
            if value in likelihoods[cls][feature]:
                posterior *= likelihoods[cls][feature][value]
            else:
                posterior *= 0  # Laplace smoothing could be added, but for simplicity
        posteriors[cls] = posterior
    return max(posteriors, key=posteriors.get)

# Test prediction on a sample
sample = X_test.iloc[0].to_dict()
print(f"\nSample: {sample}")
print(f"Actual: {y_test.iloc[0]}")
print(f"Predicted: {predict_naive_bayes(sample, priors, likelihoods, classes)}")


Sample: {'Outlook': 'Rainy', 'Temperature': 'Mild', 'Humidity': 'Normal', 'Windy': False}
Actual: Yes
Predicted: Yes


In [25]:
# Compute accuracy on test set
y_pred = []
for i in range(len(X_test)):
    sample = X_test.iloc[i].to_dict()
    pred = predict_naive_bayes(sample, priors, likelihoods, classes)
    y_pred.append(pred)

correct = sum(1 for actual, pred in zip(y_test, y_pred) if actual == pred)
accuracy = correct / len(y_test)
print(f"\nAccuracy on test set: {accuracy * 100:.2f}%")

# Display predictions
print("\nPredictions:")
for i in range(len(X_test)):
    print(f"Sample {i+1}: Actual={y_test.iloc[i]}, Predicted={y_pred[i]}")


Accuracy on test set: 100.00%

Predictions:
Sample 1: Actual=Yes, Predicted=Yes
Sample 2: Actual=Yes, Predicted=Yes
Sample 3: Actual=No, Predicted=No
Sample 4: Actual=Yes, Predicted=Yes


In [26]:
# Test with a new sample
new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Windy': True
}

prediction = predict_naive_bayes(new_sample, priors, likelihoods, classes)
print(f"\nPrediction for new sample {new_sample}: {prediction}")

# Another new sample
new_sample2 = {
    'Outlook': 'Overcast',
    'Temperature': 'Hot',
    'Humidity': 'Normal',
    'Windy': False
}

prediction2 = predict_naive_bayes(new_sample2, priors, likelihoods, classes)
print(f"Prediction for another new sample {new_sample2}: {prediction2}")


Prediction for new sample {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Windy': True}: No
Prediction for another new sample {'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'Normal', 'Windy': False}: Yes
